# Portability Round-Trips for GeneratorFamily

This notebook checks whether portability is merely schema-valid or actually semantics-preserving in downstream use.

We compare both a hand-built known translation generator and a discovered translation generator through:

- manifest export
- JSON serialization
- coercion back into canonical `GeneratorFamily`
- span comparison
- symbolic rendering
- translation-canonical discovery-input equivalence


In [ ]:
import importlib.util
import json

if importlib.util.find_spec("pysindy") is None:
    raise RuntimeError("Install pdelie[downstream] or pdelie[test] to run this notebook.")

import numpy as np

from pdelie import GeneratorFamily
from pdelie.data import generate_heat_1d_field_batch
from pdelie.discovery import build_translation_canonical_discovery_inputs
from pdelie.portability import coerce_generator_family, export_generator_family_manifest
from pdelie.residuals import HeatResidualEvaluator
from pdelie.symmetry import compare_generator_spans, fit_translation_generator, render_generator_family


In [ ]:
def make_known_translation_generator() -> GeneratorFamily:
    basis_spec = {
        "variables": ["t", "x", "u"],
        "component_names": ["xi"],
        "basis_terms": [
            {"label": "1", "powers": [0, 0, 0]},
            {"label": "t", "powers": [1, 0, 0]},
            {"label": "x", "powers": [0, 1, 0]},
            {"label": "u", "powers": [0, 0, 1]},
        ],
        "component_ordering": ["xi"],
        "term_ordering": ["1", "t", "x", "u"],
        "layout": "component_major",
    }
    return GeneratorFamily(
        parameterization="polynomial_translation_affine",
        coefficients=np.array([[1.0, 0.0, 0.0, 0.0]], dtype=float),
        basis_spec=basis_spec,
        normalization="l2_unit",
        diagnostics={},
    )


def round_trip(generator):
    manifest = export_generator_family_manifest(generator)
    restored = coerce_generator_family(json.loads(json.dumps(manifest)))
    return manifest, restored


In [ ]:
field = generate_heat_1d_field_batch(batch_size=4, num_times=33, num_points=64, seed=300)
discovered = fit_translation_generator(field, HeatResidualEvaluator())
known = make_known_translation_generator()

families = {
    "known": known,
    "discovered": discovered,
}


In [ ]:
for label, generator in families.items():
    manifest, restored = round_trip(generator)
    span_report = compare_generator_spans(generator, restored)

    print(f"=== {label} ===")
    print("manifest version:", manifest["manifest_version"])
    print("canonical dict preserved:", generator.to_dict() == restored.to_dict())
    print("principal angles:", span_report["principal_angles_radians"])
    print("projection residual:", span_report["projection_residual"]["summary"])
    print("original rendering:")
    for line in render_generator_family(generator):
        print("  ", line)
    print("round-tripped rendering:")
    for line in render_generator_family(restored):
        print("  ", line)
    print()


## Downstream semantic preservation check

For each family, we feed the original and round-tripped versions into `build_translation_canonical_discovery_inputs(...)` and compare the returned alignment metadata and trajectories.


In [ ]:
for label, generator in families.items():
    _, restored = round_trip(generator)

    original_inputs = build_translation_canonical_discovery_inputs(field, generator_family=generator)
    restored_inputs = build_translation_canonical_discovery_inputs(field, generator_family=restored)

    print(f"=== {label} downstream check ===")
    print("generator metadata equal:", original_inputs["generator_metadata"] == restored_inputs["generator_metadata"])
    print("feature names equal:", original_inputs["feature_names"] == restored_inputs["feature_names"])
    print("alignment shifts allclose:", np.allclose(original_inputs["alignment_shifts"], restored_inputs["alignment_shifts"]))
    print("time values allclose:", np.allclose(original_inputs["time_values"], restored_inputs["time_values"]))
    print(
        "trajectory blocks allclose:",
        all(np.allclose(a, b) for a, b in zip(original_inputs["trajectories"], restored_inputs["trajectories"])),
    )
    print()
